# Data Preparation

In [1]:
import numpy as np
import pandas as pd
from datasets import Dataset
from utils.hub_card import push_dataset_card
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
pd.set_option('display.max_columns', 200)

RAW_DATA_REPO = 'ADS599-Capstone/raw_data'
MODELING_REPO = 'ADS599-Capstone/modeling_data'
RANDOM_STATE = 42

# Cohort labels used for training Part 1
TRAIN_LABELS = [
    'ED_DISCHARGE_STABLE',
    'ED_DIRECT_ICU',
    'ED_WARD_DISCHARGE',
    'ED_WARD_ICU',
]

## Load raw data from HuggingFace
Loading:
- `full_patient_state` — the preprocessed timestep-level patient state (from modeling_data repo)
- `cohort_base` — cohort labels, demographics, triage vitals, chief complaint
- `cohort_with_triage` — includes chief complaint text

In [3]:
df_state = load_dataset(
    path=MODELING_REPO,
    name='full_patient_state',
    split='full_patient_state',
    verification_mode='no_checks'
).to_pandas()
print(f'  shape: {df_state.shape}')
print(f'  unique stays: {df_state["ed_stay_id"].nunique():,}')

df_cohort = load_dataset(
    path=RAW_DATA_REPO,
    name='cohort_base',
    split='base',
    verification_mode='no_checks'
).to_pandas()
print(f'  shape: {df_cohort.shape}')
print(f'  cohort_label distribution:')
print(df_cohort['cohort_label'].value_counts())

df_triage = load_dataset(
    path=RAW_DATA_REPO,
    name='cohort_with_triage',
    split='with_triage',
    verification_mode='no_checks'
).to_pandas()
print(f'  shape: {df_triage.shape}')
print(f'  chief complaint null: {df_triage["chiefcomplaint"].isna().sum():,}')
print(f'  unique chief complaints: {df_triage["chiefcomplaint"].nunique():,}')

README.md: 0.00B [00:00, ?B/s]

Generating full_patient_state split:   0%|          | 0/6551723 [00:00<?, ? examples/s]

  shape: (6551723, 136)
  unique stays: 84,211
  shape: (399573, 24)
  cohort_label distribution:
cohort_label
ED_DISCHARGE_STABLE        241206
ED_WARD_DISCHARGE          126507
ED_DIRECT_ICU               22993
ED_WARD_ICU                  8491
ED_DISCHARGE_RETURN_ICU       304
ED_DISCHARGE_DIED_72H          72
Name: count, dtype: int64
  shape: (397675, 36)
  chief complaint null: 8
  unique chief complaints: 57,578


## Apply existing preprocessing from modeling.ipynb
Replicating the preprocessing already done — gender encoding, acuity cast, missing masks, one-hot encoding.
This keeps the data consistent with what was used in the RL notebook.

In [4]:
df = df_state.copy()

# Gender
gender_map = {'F': 1, 'M': 0}
df['gender'] = df['gender'].map(gender_map)

# Acuity
df['acuity'] = pd.to_numeric(df['acuity'], errors='coerce')

# Height / weight missing flags
df['height_missing'] = df['height'].isna().astype(int)
df['weight_missing'] = df['weight'].isna().astype(int)
df[['height', 'weight']] = df[['height', 'weight']].fillna(0)

# Pain
df['pain_missing'] = (df['current_pain'] == 'Other').astype(int)
df['current_pain'] = pd.to_numeric(df['current_pain'], errors='coerce').fillna(0)

# Admission type and arrival transport one-hot
df['admission_missing'] = df['admission_type'].isna().astype(int)
at_dummies = pd.get_dummies(df['admission_type'], prefix='admission_type', dummy_na=False, dtype=int)
arrival_dummies = pd.get_dummies(df['arrival_transport'], prefix='arrival_transport', dtype=int)
df = pd.concat([df, at_dummies, arrival_dummies], axis=1).drop(
    columns=['admission_type', 'arrival_transport']
)

print(f'Preprocessed shape: {df.shape}')
print(f'Columns added: {list(at_dummies.columns) + list(arrival_dummies.columns)}')

Preprocessed shape: (6551723, 152)
Columns added: ['admission_type_AMBULATORY OBSERVATION', 'admission_type_DIRECT EMER.', 'admission_type_DIRECT OBSERVATION', 'admission_type_ELECTIVE', 'admission_type_EU OBSERVATION', 'admission_type_EW EMER.', 'admission_type_OBSERVATION ADMIT', 'admission_type_SURGICAL SAME DAY ADMISSION', 'admission_type_URGENT', 'arrival_transport_AMBULANCE', 'arrival_transport_HELICOPTER', 'arrival_transport_OTHER', 'arrival_transport_UNKNOWN', 'arrival_transport_WALK IN']


## Resample from 30-minute to 1-hour intervals

The raw data has one row per 30 minutes. We collapse to 1-hour bins.

**Aggregation rules by column type:**
- **Continuous vitals** (`current_*`, rolling stats): `last` — the value at the end of the hour.
  A heart rate spike at minute 55 matters clinically even if it was normal for the first 55 minutes.
  Mean would smooth out those spikes.
- **Binary / count flags** (labs, micro, meds, actions, ECG, radiology): `max` — if either
  30-min window had the event, the 1-hour row should reflect it.
- **Static columns** (demographics, recon_*, one-hots, terminal_event, cohort_label): `first` —
  these don't change within a stay so any row gives the same value.
- **Time**: floor to the hour (used as the new index key).

In [5]:
# Column classification for aggregation
vital_last_cols = [
    c for c in df.columns if c.startswith('current_')
] + ['time_since_last_hrs', 'ecg_status', 'rad_status']
vital_last_cols = [c for c in vital_last_cols if c in df.columns]

# Rolling stats
rolling_cols = [
    c for c in df.columns
    if any(c.endswith(x) for x in ['_rolling2h', '_delta', '_rate'])
]

# Binary / count flags with MAX
binary_count_cols = [
    c for c in df.columns if c in [
        # Lab orders
        'Chemistry-Blood', 'Hematology-Blood', 'Hematology-Urine', 'Chemistry-Urine',
        'Blood Gas-Blood', 'Hematology-Cerebrospinal Fluid', 'Hematology-Ascites',
        'Chemistry-Other Body Fluid', 'Hematology-Pleural', 'Hematology-Joint Fluid',
        'Chemistry-Pleural', 'Chemistry-Ascites', 'Chemistry-Cerebrospinal Fluid',
        'Hematology-Other Body Fluid', 'Chemistry-Stool', 'Hematology-Bone Marrow',
        'Blood Gas-Other Body Fluid', 'Hematology-Stool', 'Chemistry-Joint Fluid',
        # Microbiology cultures
        'URINE', 'BLOOD CULTURE', 'STOOL', 'SWAB', 'TISSUE', 'PLEURAL FLUID',
        'CSF;SPINAL FLUID', 'SPUTUM', 'BRONCHOALVEOLAR LAVAGE', 'JOINT FLUID',
        'ABSCESS', 'FLUID,OTHER', 'OTHER', 'IMMUNOLOGY', 'SEROLOGY/BLOOD',
        'MRSA SCREEN', 'Rapid Respiratory Viral Screen & Culture', 'PERITONEAL FLUID',
        'Blood (CMV AB)', 'Blood (EBV)', 'BILE',
        # ED medications dispensed
        'Analgesic - Opioid/NSAID', 'Antiemetic', 'Analgesic - Acetaminophen',
        'Antibiotic', 'Benzodiazepine - Sedative/Anxiolytic', 'Analgesic - NSAID',
        'Bronchodilator', 'Antiplatelet', 'GI - Acid Suppression', 'Corticosteroid',
        'Beta Blocker', 'Diuretic', 'Antipsychotic', 'Anticoagulant',
        'Insulin/Glucose', 'Calcium Channel Blocker', 'Nitrate', 'ACE Inhibitor',
        'IV Fluid', 'Anticonvulsant', 'Antiarrhythmic',
        # Action flags
        'observe', 'vitals_checked', 'labs_ordered', 'micro_ordered',
        'ecg_ordered', 'rad_ordered', 'dispense_meds', 'ward_transfer',
        'discharge', 'transfer_icu',
        # Episode state
        'in_ed', 'in_ward', 'ed_boarding',
    ]
]

# Static columns — same value for all rows in a stay, use FIRST
# Everything not already classified
already_classified = set(vital_last_cols + rolling_cols + binary_count_cols + ['ed_stay_id', 'subject_id', 'hadm_id', 'time'])
static_cols = [c for c in df.columns if c not in already_classified]

print(f'vital_last_cols:    {len(vital_last_cols)}')
print(f'rolling_cols:       {len(rolling_cols)}')
print(f'binary_count_cols:  {len(binary_count_cols)}')
print(f'static_cols:        {len(static_cols)}')
print(f'Total accounted for: {len(vital_last_cols)+len(rolling_cols)+len(binary_count_cols)+len(static_cols)+4} / {len(df.columns)}')

vital_last_cols:    12
rolling_cols:       18
binary_count_cols:  74
static_cols:        44
Total accounted for: 152 / 152


In [6]:
# Resample to 1-hour bins
df['time'] = pd.to_datetime(df['time'])

# Floor time to the nearest hour to create the 1-hour bin key
df['time_1h'] = df['time'].dt.floor('h')

print(f'Rows before resample: {len(df):,}')
print(f'Unique stays before:  {df["ed_stay_id"].nunique():,}')

# Aggregation dict
agg_dict = {}
for c in vital_last_cols:
    if c in df.columns:
        agg_dict[c] = 'last'
for c in rolling_cols:
    if c in df.columns:
        agg_dict[c] = 'last'
for c in binary_count_cols:
    if c in df.columns:
        agg_dict[c] = 'max'
for c in static_cols:
    if c in df.columns and c not in ('ed_stay_id', 'subject_id', 'hadm_id', 'time', 'time_1h'):
        agg_dict[c] = 'first'

# Group by stay + 1-hour bin and aggregate
df_hourly = (
    df.groupby(['ed_stay_id', 'subject_id', 'time_1h'], sort=True)
    .agg(agg_dict)
    .reset_index()
    .rename(columns={'time_1h': 'time'})
)

print(f'\nRows after resample:  {len(df_hourly):,}')
print(f'Unique stays after:   {df_hourly["ed_stay_id"].nunique():,}')
print(f'Expected reduction:   ~{len(df)//2:,} (50% for clean 30min→1h)')
print(f'Actual reduction:     {len(df)-len(df_hourly):,} rows removed ({(1-len(df_hourly)/len(df))*100:.1f}%)')

# Replace df with the hourly version for all downstream cells
df = df_hourly
print(f'\ndf is now hourly. Shape: {df.shape}')

Rows before resample: 6,551,723
Unique stays before:  84,211

Rows after resample:  3,307,296
Unique stays after:   84,211
Expected reduction:   ~3,275,861 (50% for clean 30min→1h)
Actual reduction:     3,244,427 rows removed (49.5%)

df is now hourly. Shape: (3307296, 151)


In [7]:
# Sanity checks

# Row count should be roughly half
rows_per_stay = df.groupby('ed_stay_id').size()
print('Timesteps per stay after resampling:')
print(rows_per_stay.describe().round(1))

# Static columns should have zero variance within a stay
static_check_cols = ['terminal_event', 'gender', 'anchor_age']
static_check_cols = [c for c in static_check_cols if c in df.columns]
for col in static_check_cols:
    nunique_per_stay = df.groupby('ed_stay_id')[col].nunique()
    bad = (nunique_per_stay > 1).sum()
    print(f'{col}: stays with >1 unique value = {bad} (should be 0)')

# Check that no future timestep information bled into earlier hours
is_sorted = df.groupby('ed_stay_id')['time'].apply(lambda x: x.is_monotonic_increasing).all()
print(f'\nTime is monotonically increasing within all stays: {is_sorted}')

# Spot check one stay — show original 30-min vs new 1-hour rows
sample_stay = df['ed_stay_id'].iloc[0]
print(f'\nSpot check stay {sample_stay}:')
print(df[df['ed_stay_id'] == sample_stay][['time', 'current_heartrate', 'current_o2sat', 'labs_ordered']].head(6))

Timesteps per stay after resampling:
count    84211.0
mean        39.3
std         85.0
min          1.0
25%          5.0
50%          8.0
75%         41.0
max       5626.0
dtype: float64
terminal_event: stays with >1 unique value = 0 (should be 0)
gender: stays with >1 unique value = 0 (should be 0)
anchor_age: stays with >1 unique value = 0 (should be 0)

Time is monotonically increasing within all stays: True

Spot check stay 30000012:
                 time  current_heartrate  current_o2sat  labs_ordered
0 2126-02-14 20:00:00               96.0           93.0             0
1 2126-02-14 21:00:00               96.0           93.0             1
2 2126-02-14 22:00:00               96.0           93.0             1
3 2126-02-14 23:00:00               80.0           99.0             0
4 2126-02-15 00:00:00               88.0          100.0             0
5 2126-02-15 01:00:00               88.0          100.0             0


## Join cohort label
The 6-class cohort label is the target variable for Part 1

In [8]:
df = df.merge(
    df_cohort[['ed_stay_id', 'cohort_label']],
    on='ed_stay_id',
    how='left'
)

# Verify join
null_labels = df['cohort_label'].isna().sum()
print(f'Null cohort_label after join: {null_labels:,}')
if null_labels > 0:
    print('WARNING: some stays did not match cohort — check ed_stay_id alignment')

print('\ncohort_label distribution in merged data (stay level):')
print(df.drop_duplicates('ed_stay_id')['cohort_label'].value_counts())

Null cohort_label after join: 0

cohort_label distribution in merged data (stay level):
cohort_label
ED_DISCHARGE_STABLE        43986
ED_WARD_DISCHARGE          21816
ED_DIRECT_ICU              11442
ED_WARD_ICU                 6723
ED_DISCHARGE_RETURN_ICU      209
ED_DISCHARGE_DIED_72H         35
Name: count, dtype: int64


## Add chief complaint
Chief complaint is not in full_patient_state. It comes from cohort_with_triage.

### Encoding strategy:
Raw chief complaint has hundreds of unique free-text values — one-hot encoding every value would create hundreds of near-zero columns and cause overfitting.
We use two encodings and keep both:
1. **Clinical category grouping** (~20 groups) — interpretable, good for LR and RF
2. **TF-IDF** (top 50 terms) — captures signal from less common complaints, good for XGBoost

In [9]:
# Get chief complaint from triage table (one per stay)
cc = df_triage[['ed_stay_id', 'chiefcomplaint']].drop_duplicates('ed_stay_id').copy()
cc['chiefcomplaint'] = cc['chiefcomplaint'].fillna('unknown').str.lower().str.strip()

# Clinical category mapping
# Maps common chief complaint keywords to ~20 clinical categories.
# Order matters: first match wins.
CC_CATEGORIES = [
    ('chest pain|chest tightness|chest pressure|cp ',        'chest_pain'),
    ('shortness of breath|dyspnea|sob|difficulty breathing', 'dyspnea'),
    ('altered mental|confusion|altered loc|ams|unresponsive','altered_mental_status'),
    ('syncope|syncopal|fainting|passed out|loss of consciousness', 'syncope'),
    ('abdominal pain|abd pain|stomach pain|abdominal cramp', 'abdominal_pain'),
    ('nausea|vomiting|n/v|emesis',                           'nausea_vomiting'),
    ('fever|febrile|temperature',                            'fever'),
    ('headache|head pain|migraine',                          'headache'),
    ('back pain|lower back|lumbar',                          'back_pain'),
    ('fall|fell|mechanical fall',                            'fall'),
    ('trauma|injury|mvc|motor vehicle|accident',             'trauma'),
    ('seizure|convulsion',                                   'seizure'),
    ('stroke|cva|facial droop|weakness one side|slurred',   'stroke_like'),
    ('palpitation|heart racing|tachycardia',                 'palpitations'),
    ('weakness|fatigue|malaise',                             'weakness_fatigue'),
    ('urinary|dysuria|frequency|uti',                        'urinary'),
    ('leg pain|leg swelling|dvt|calf pain',                  'leg_complaint'),
    ('psychiatric|suicidal|overdose|ingestion',              'psychiatric_overdose'),
    ('bleeding|hemorrhage|blood',                            'bleeding'),
    ('pain',                                                 'pain_other')
]

def map_chief_complaint(text):
    for pattern, category in CC_CATEGORIES:
        if pd.notna(text) and __import__('re').search(pattern, str(text)):
            return category
    return 'other'

cc['cc_category'] = cc['chiefcomplaint'].apply(map_chief_complaint)

print('Chief complaint category distribution:')
print(cc['cc_category'].value_counts())
print(f'\nCategory coverage: {(cc["cc_category"] != "other").mean()*100:.1f}%')

Chief complaint category distribution:
cc_category
other                    127745
abdominal_pain            43607
pain_other                40823
chest_pain                27676
fall                      20086
dyspnea                   19902
trauma                    15345
nausea_vomiting           15096
back_pain                 13558
headache                  10633
fever                     10509
weakness_fatigue           8834
altered_mental_status      8715
leg_complaint              7865
syncope                    7345
urinary                    4759
palpitations               4326
seizure                    3402
bleeding                   3305
stroke_like                2604
psychiatric_overdose       1540
Name: count, dtype: int64

Category coverage: 67.9%


In [10]:
# One-hot encode clinical categories
cc_dummies = pd.get_dummies(cc['cc_category'], prefix='cc', dtype=int)
cc = pd.concat([cc[['ed_stay_id', 'chiefcomplaint']], cc_dummies], axis=1)

# TF-IDF on raw text (top 50 terms)
tfidf = TfidfVectorizer(max_features=50, stop_words='english', ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(cc['chiefcomplaint'].fillna('unknown'))
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=[f'cc_tfidf_{t}' for t in tfidf.get_feature_names_out()]
)
tfidf_df['ed_stay_id'] = cc['ed_stay_id'].values

# Merge both encodings onto df (timestep level)
df = df.merge(cc[['ed_stay_id'] + list(cc_dummies.columns)], on='ed_stay_id', how='left')
df = df.merge(tfidf_df, on='ed_stay_id', how='left')

cc_cols   = list(cc_dummies.columns)
tfidf_cols = [c for c in df.columns if c.startswith('cc_tfidf_')]

print(f'Clinical category columns added: {len(cc_cols)}')
print(f'TF-IDF columns added:            {len(tfidf_cols)}')
print(f'Updated df shape: {df.shape}')

Clinical category columns added: 21
TF-IDF columns added:            50
Updated df shape: (3307296, 223)


## Define feature sets
### Part 1 (classification) — TRIAGE-TIME ONLY features
These are features available at the moment the patient arrives. No in-stay labs, no medications dispensed during the visit, no imaging during the visit.
Including those would be leakage — we'd be using information that didn't exist
at the time the prediction is supposed to be made.

### Part 2 (time series) — ALL features including dynamic
The full timestep feature set (same as used in the RL notebook).

In [ ]:
# PART 1: Triage-time features (no leakage)

# Demographics
demo_cols = ['gender', 'anchor_age', 'height', 'weight',
             'height_missing', 'weight_missing']

# Triage vitals (first measurement only — use triage values, not rolling/delta)
triage_vital_cols = [
    'current_temperature', 'current_heartrate', 'current_resprate',
    'current_o2sat', 'current_sbp', 'current_dbp', 'current_pain',
    'current_pulse_pressure', 'current_map', 'pain_missing'
]

# Triage metadata
triage_meta_cols = ['acuity']

# Arrival info
arrival_col_list = [c for c in df.columns if c.startswith('arrival_transport_')]

# Pre-visit medications (set at arrival from medrecon)
recon_cols = [c for c in df.columns if c.startswith('recon_')]

# Chief complaint (recorded at triage)
# Use category encoding for interpretable models, TF-IDF for XGBoost
cc_category_cols = [c for c in df.columns if c.startswith('cc_') and not c.startswith('cc_tfidf_')]
cc_tfidf_cols = [c for c in df.columns if c.startswith('cc_tfidf_')]

# Columns that are LEAKAGE for Part 1 (in-stay dynamic features)
# Lab orders during the stay
lab_dynamic_cols = [c for c in df.columns if '-' in c and any(
    x in c for x in ['Chemistry', 'Hematology', 'Blood Gas']
)]
# Micro culture orders during the stay
micro_dynamic_cols = ['URINE', 'BLOOD CULTURE', 'STOOL', 'SWAB', 'TISSUE',
                      'PLEURAL FLUID', 'CSF;SPINAL FLUID', 'SPUTUM',
                      'BRONCHOALVEOLAR LAVAGE', 'JOINT FLUID', 'ABSCESS',
                      'FLUID,OTHER', 'OTHER', 'IMMUNOLOGY', 'SEROLOGY/BLOOD',
                      'MRSA SCREEN', 'Rapid Respiratory Viral Screen & Culture',
                      'PERITONEAL FLUID', 'Blood (CMV AB)', 'Blood (EBV)', 'BILE']
micro_dynamic_cols = [c for c in micro_dynamic_cols if c in df.columns]
# Medications dispensed during the visit
ed_med_cols = [c for c in df.columns if c in [
    'Analgesic - Opioid/NSAID', 'Antiemetic', 'Analgesic - Acetaminophen',
    'Antibiotic', 'Benzodiazepine - Sedative/Anxiolytic', 'Analgesic - NSAID',
    'Bronchodilator', 'Antiplatelet', 'GI - Acid Suppression', 'Corticosteroid',
    'Beta Blocker', 'Diuretic', 'Antipsychotic', 'Anticoagulant',
    'Insulin/Glucose', 'Calcium Channel Blocker', 'Nitrate', 'ACE Inhibitor',
    'IV Fluid', 'Anticonvulsant', 'Antiarrhythmic'
]]
# ECG and radiology ordered during the stay
ecg_rad_cols = ['ecg_status', 'rad_status']
# Rolling/delta vitals
rolling_delta_cols = [c for c in df.columns if any(
    c.endswith(x) for x in ['_rolling2h', '_delta', '_rate']
)]
# Action and episode state columns
action_episode_cols = [
    'observe', 'vitals_checked', 'labs_ordered', 'micro_ordered',
    'ecg_ordered', 'rad_ordered', 'dispense_meds', 'ward_transfer',
    'discharge', 'transfer_icu', 'in_ed', 'in_ward', 'ed_boarding',
    'time_since_last_hrs'
]

# Final Part 1 feature set (triage-time, no leakage)
PART1_FEATURES = (
    demo_cols +
    triage_vital_cols +
    triage_meta_cols +
    arrival_col_list +
    admission_col_list +
    recon_cols +
    cc_category_cols
)
# For XGBoost, swap cc_category_cols for TF-IDF
PART1_FEATURES_XGBOOST = (
    demo_cols +
    triage_vital_cols +
    triage_meta_cols +
    arrival_col_list +
    admission_col_list +
    recon_cols +
    cc_tfidf_cols
)

# Part 2 feature set (same as RL state_cols which includes dynamic features)
vitals_all = [c for c in df.columns
              if c.startswith('current_') or c in ['time_since_last_hrs']]
PART2_FEATURES = (
    demo_cols + triage_vital_cols + triage_meta_cols +
    arrival_col_list + admission_col_list + recon_cols + cc_category_cols +
    lab_dynamic_cols + micro_dynamic_cols + ed_med_cols + ecg_rad_cols +
    rolling_delta_cols + ['time_since_last_hrs']
)

# Remove duplicates while preserving order
PART1_FEATURES = list(dict.fromkeys([c for c in PART1_FEATURES if c in df.columns]))
PART1_FEATURES_XGBOOST = list(dict.fromkeys([c for c in PART1_FEATURES_XGBOOST if c in df.columns]))
PART2_FEATURES = list(dict.fromkeys([c for c in PART2_FEATURES if c in df.columns]))

print(f'Part 1 features (triage-time, interpretable models): {len(PART1_FEATURES)}')
print(f'Part 1 features (triage-time, XGBoost with TF-IDF):  {len(PART1_FEATURES_XGBOOST)}')
print(f'Part 2 features (full dynamic state):                {len(PART2_FEATURES)}')

Part 1 features (triage-time, interpretable models): 72
Part 1 features (triage-time, XGBoost with TF-IDF):  101
Part 2 features (full dynamic state):                154


## Part 1 dataset (one row per stay, triage-time features only)
Take the first timestep per stay, triage time.
Drop all in-stay dynamic features.

In [12]:
# Take the first timestep per stay
df_sorted = df.sort_values(['ed_stay_id', 'time'])
df_triage_snap = df_sorted.groupby('ed_stay_id').first().reset_index()

# Keep only Part 1 features + identifiers + label
KEEP_COLS = ['ed_stay_id', 'subject_id', 'cohort_label'] + PART1_FEATURES
KEEP_COLS = [c for c in KEEP_COLS if c in df_triage_snap.columns]
df_part1 = df_triage_snap[KEEP_COLS].copy()

# Verify no missing labels
print(f'Part 1 dataset shape: {df_part1.shape}')
print(f'Null cohort_label: {df_part1["cohort_label"].isna().sum()}')
print('\nLabel distribution:')
print(df_part1['cohort_label'].value_counts())

# Sanity check: dynamic columns should NOT be in df_part1
leaked_cols = [c for c in lab_dynamic_cols + micro_dynamic_cols + ed_med_cols
               if c in df_part1.columns]
print(f'\nLeakage check — in-stay dynamic columns present: {len(leaked_cols)}')
if leaked_cols:
    print(f'  WARNING: {leaked_cols}')
else:
    print('  PASS — no in-stay features in Part 1 dataset')

Part 1 dataset shape: (84211, 75)
Null cohort_label: 0

Label distribution:
cohort_label
ED_DISCHARGE_STABLE        43986
ED_WARD_DISCHARGE          21816
ED_DIRECT_ICU              11442
ED_WARD_ICU                 6723
ED_DISCHARGE_RETURN_ICU      209
ED_DISCHARGE_DIED_72H         35
Name: count, dtype: int64

Leakage check — in-stay dynamic columns present: 0
  PASS — no in-stay features in Part 1 dataset


## Train/test split for Part 1

- Split by **subject_id** — prevents leakage for patients with multiple visits
- Stratify by **cohort_label** — preserves class distribution across all 4 classes
- 4 main labels used for both training and testing (80/20 split)

In [13]:
# Filter to the 4 main labels only
df_train_pool = df_part1[df_part1['cohort_label'].isin(TRAIN_LABELS)].copy()

print(f'Training pool (4 main labels): {len(df_train_pool):,} stays')

# Get unique subjects in training pool for splitting
# One subject may have multiple stays so split subjects, not stays
subjects = df_train_pool[['subject_id', 'ed_stay_id', 'cohort_label']].drop_duplicates()

# For subjects with multiple stays use the most severe label to stratify
severity_order = {
    'ED_DIRECT_ICU': 0,
    'ED_WARD_ICU': 1,
    'ED_WARD_DISCHARGE': 2,
    'ED_DISCHARGE_STABLE': 3,
}
subjects['severity'] = subjects['cohort_label'].map(severity_order)
subject_label = (subjects.sort_values('severity')
                         .drop_duplicates('subject_id', keep='first')[['subject_id', 'cohort_label']])

# Split subjects 80/20 stratified by cohort label
train_subjects, test_subjects = train_test_split(
    subject_label['subject_id'],
    test_size=0.2,
    stratify=subject_label['cohort_label'],
    random_state=RANDOM_STATE
)

train_stays = df_train_pool[df_train_pool['subject_id'].isin(train_subjects)]
test_stays = df_train_pool[df_train_pool['subject_id'].isin(test_subjects)]

overlap = set(train_stays['subject_id']) & set(test_stays['subject_id'])
print(f'\nSubject overlap between train and test: {len(overlap)} (should be 0)')

print(f'\nTrain stays: {len(train_stays):,}')
print(train_stays['cohort_label'].value_counts())
print(f'\nTest stays: {len(test_stays):,}')
print(test_stays['cohort_label'].value_counts())

Training pool (4 main labels): 83,967 stays

Subject overlap between train and test: 0 (should be 0)

Train stays: 67,085
cohort_label
ED_DISCHARGE_STABLE    35156
ED_WARD_DISCHARGE      17414
ED_DIRECT_ICU           9137
ED_WARD_ICU             5378
Name: count, dtype: int64

Test stays: 16,882
cohort_label
ED_DISCHARGE_STABLE    8830
ED_WARD_DISCHARGE      4402
ED_DIRECT_ICU          2305
ED_WARD_ICU            1345
Name: count, dtype: int64


## Scale numerical features

In [14]:
# Columns to scale: continuous numerical features
SCALE_COLS = [
    'anchor_age', 'height', 'weight',
    'current_temperature', 'current_heartrate', 'current_resprate',
    'current_o2sat', 'current_sbp', 'current_dbp', 'current_pain',
    'current_pulse_pressure', 'current_map', 'acuity'
]
SCALE_COLS = [c for c in SCALE_COLS if c in PART1_FEATURES]

scaler = StandardScaler()

X_train = train_stays[PART1_FEATURES].copy()
X_test  = test_stays[PART1_FEATURES].copy()

y_train = train_stays['cohort_label'].values
y_test  = test_stays['cohort_label'].values

# Fit on train only
X_train[SCALE_COLS] = scaler.fit_transform(X_train[SCALE_COLS])
X_test[SCALE_COLS] = scaler.transform(X_test[SCALE_COLS])

# Handle remaining nulls
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

print(f'X_train: {X_train.shape}')
print(pd.Series(y_train).value_counts())
print(f'\nX_test: {X_test.shape}')
print(pd.Series(y_test).value_counts())
print(f'\nNull check — X_train nulls: {X_train.isna().sum().sum()}')

X_train: (67085, 72)
ED_DISCHARGE_STABLE    35156
ED_WARD_DISCHARGE      17414
ED_DIRECT_ICU           9137
ED_WARD_ICU             5378
Name: count, dtype: int64

X_test: (16882, 72)
ED_DISCHARGE_STABLE    8830
ED_WARD_DISCHARGE      4402
ED_DIRECT_ICU          2305
ED_WARD_ICU            1345
Name: count, dtype: int64

Null check — X_train nulls: 0


## Encode target label for sklearn models

In [15]:
le = LabelEncoder()
le.fit(TRAIN_LABELS)

y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

print('Label encoding:')
for i, label in enumerate(le.classes_):
    print(f'  {i}: {label}')

Label encoding:
  0: ED_DIRECT_ICU
  1: ED_DISCHARGE_STABLE
  2: ED_WARD_DISCHARGE
  3: ED_WARD_ICU


## Part 2 dataset — timestep-level with escalation label
For the time series escalation problem:
- Keep all timestep rows
- Add binary `escalation_needed` label: 1 if the stay ended in ICU, 0 otherwise
- The label is stay-level (same for all rows in a stay) but the features vary by timestep
- Rolling window aggregation produces a 'state so far' snapshot at each timestep

In [16]:
# Binary escalation label
# 1 = patient ended up in ICU (either directly or via ward)
# 0 = patient was discharged (ED stable or ward discharge)
# Rare labels (bounce-back, died) excluded from Part 2 training same as Part 1
icu_labels = {'ED_DIRECT_ICU', 'ED_WARD_ICU'}

df_ts = df[df['cohort_label'].isin(TRAIN_LABELS)].copy()
df_ts['escalation_needed'] = df_ts['cohort_label'].isin(icu_labels).astype(int)

print(f'Part 2 dataset: {len(df_ts):,} timestep rows')
print(f'Unique stays:   {df_ts["ed_stay_id"].nunique():,}')
print(f'\nEscalation label distribution (stay level):')
print(df_ts.drop_duplicates('ed_stay_id')['escalation_needed'].value_counts())
print(f'  ({df_ts.drop_duplicates("ed_stay_id")["escalation_needed"].mean()*100:.1f}% need escalation)')

Part 2 dataset: 3,304,634 timestep rows
Unique stays:   83,967

Escalation label distribution (stay level):
escalation_needed
0    65802
1    18165
Name: count, dtype: int64
  (21.6% need escalation)


In [17]:
# Rolling window feature aggregation
# At each timestep t, aggregate all observations from t=0 to t.
# This creates a 'state so far' snapshot that avoids using future information.
# Features: mean, max, min, last value of each vital sign up to time t.
# This is the simplest approach — no LSTM needed, same sklearn models as Part 1.

ROLLING_VITALS = [
    'current_temperature', 'current_heartrate', 'current_resprate',
    'current_o2sat', 'current_sbp', 'current_dbp', 'current_pain',
    'current_pulse_pressure', 'current_map'
]
ROLLING_VITALS = [c for c in ROLLING_VITALS if c in df_ts.columns]

df_ts = df_ts.sort_values(['ed_stay_id', 'time'])

agg_frames = []
for vital in ROLLING_VITALS:
    grp = df_ts.groupby('ed_stay_id')[vital]
    df_ts[f'{vital}_cummax'] = grp.cummax()
    df_ts[f'{vital}_cummin'] = grp.cummin()
    df_ts[f'{vital}_cummean'] = grp.expanding().mean().values

# Minutes into stay
df_ts['time'] = pd.to_datetime(df_ts['time'])
df_ts['minutes_into_stay'] = (
    df_ts.groupby('ed_stay_id')['time']
    .transform(lambda x: (x - x.iloc[0]).dt.total_seconds() / 60)
)

cum_vital_cols = ([f'{v}_cummax' for v in ROLLING_VITALS] +
                  [f'{v}_cummin' for v in ROLLING_VITALS] +
                  [f'{v}_cummean' for v in ROLLING_VITALS])

PART2_TS_FEATURES = (
    demo_cols + triage_meta_cols + recon_cols + cc_category_cols +
    arrival_col_list + admission_col_list +
    ROLLING_VITALS +        # current values at this timestep
    cum_vital_cols +        # cumulative stats up to this timestep
    lab_dynamic_cols +      # lab results seen so far
    ['minutes_into_stay', 'time_since_last_hrs']
)
PART2_TS_FEATURES = list(dict.fromkeys([c for c in PART2_TS_FEATURES if c in df_ts.columns]))

print(f'Part 2 timestep feature count: {len(PART2_TS_FEATURES)}')
print(f'Cumulative vital columns added: {len(cum_vital_cols)}')

Part 2 timestep feature count: 119
Cumulative vital columns added: 27


In [18]:
# Part 2 train/test split
df_ts_train = df_ts[df_ts['subject_id'].isin(train_subjects)].copy()
df_ts_test = df_ts[df_ts['subject_id'].isin(test_subjects)].copy()

X_ts_train = df_ts_train[PART2_TS_FEATURES].fillna(0)
X_ts_test = df_ts_test[PART2_TS_FEATURES].fillna(0)
y_ts_train = df_ts_train['escalation_needed'].values
y_ts_test = df_ts_test['escalation_needed'].values

# Scale continuous cols
TS_SCALE_COLS = [c for c in SCALE_COLS + cum_vital_cols + ['minutes_into_stay', 'time_since_last_hrs']
                 if c in PART2_TS_FEATURES]
ts_scaler = StandardScaler()
X_ts_train[TS_SCALE_COLS] = ts_scaler.fit_transform(X_ts_train[TS_SCALE_COLS])
X_ts_test[TS_SCALE_COLS] = ts_scaler.transform(X_ts_test[TS_SCALE_COLS])

print(f'Part 2 train rows: {len(X_ts_train):,} | escalation rate: {y_ts_train.mean()*100:.1f}%')
print(f'Part 2 test rows:  {len(X_ts_test):,}  | escalation rate: {y_ts_test.mean()*100:.1f}%')

Part 2 train rows: 2,641,358 | escalation rate: 19.0%
Part 2 test rows:  663,276  | escalation rate: 18.3%


## Summary and save
Final check on all prepared datasets before modeling.

In [19]:
print(f'\nPART 1 — Classification (triage-time features only)')
print(f'  X_train: {X_train.shape}  — {len(PART1_FEATURES)} features')
print(f'  X_test:  {X_test.shape}')
print(f'  Target:  cohort_label')
print(f'  Leakage: None — all features from triage time only')

print(f'\nPART 2 — Time series escalation')
print(f'  X_ts_train: {X_ts_train.shape}  — {len(PART2_TS_FEATURES)} features')
print(f'  X_ts_test:  {X_ts_test.shape}')
print(f'  Target:  escalation_needed (binary, 1 = ICU outcome)')
print(f'  Leakage: None — cumulative aggregation only uses past timesteps')

print(f'\nSplit design:')
print(f'  Split by subject_id - no patient appears in both train and test')
print(f'  Stratified by cohort_label - class distribution preserved')
print(f'  Scaler fit on training data only')

# Push to HuggingFace
DEST_REPO   = "ADS599-Capstone/modeling_data"
DESCRIPTION = (
    'Preprocessed classification and time-series escalation datasets. '
    'Part 1: one row per ED stay, triage-time features only (no leakage), '
    '4-class cohort label target. '
    'Part 2: hourly timestep rows, ED-only (in_ward=0), capped at 48h, '
    'binary escalation_needed label, cumulative vital aggregation features.'
)

part2_train_full = X_ts_train.assign(
    escalation_needed=y_ts_train,
    ed_stay_id=df_ts_train['ed_stay_id'].values,
    minutes_into_stay=df_ts_train['minutes_into_stay'].values
)
part2_test_full = X_ts_test.assign(
    escalation_needed=y_ts_test,
    ed_stay_id=df_ts_test['ed_stay_id'].values,
    minutes_into_stay=df_ts_test['minutes_into_stay'].values
)

datasets_to_push = [
    (X_train.assign(cohort_label=y_train), 'part1_train', 'part1_train'),
    (X_test.assign(cohort_label=y_test), 'part1_test', 'part1_test'),
    (part2_train_full, 'part2_train', 'part2_train'),
    (part2_test_full, 'part2_test', 'part2_test'),
]

for df_push, config_name, split_name in datasets_to_push:
    print(f"  Pushing {config_name} ({df_push.shape})...")
    ds = Dataset.from_pandas(df_push.reset_index(drop=True), preserve_index=False)
    ds.push_to_hub(DEST_REPO, config_name=config_name, split=split_name)
    push_dataset_card(
        DEST_REPO,
        config_name=config_name,
        split=split_name,
        data_dir=config_name,
        description=DESCRIPTION
    )

print("\nAll datasets pushed to HuggingFace.")
print(f"Repo: {DEST_REPO}")
print('Configs: part1_train, part1_test, part2_train, part2_test')


PART 1 — Classification (triage-time features only)
  X_train: (67085, 72)  — 72 features
  X_test:  (16882, 72)
  Target:  cohort_label
  Leakage: None — all features from triage time only

PART 2 — Time series escalation
  X_ts_train: (2641358, 119)  — 119 features
  X_ts_test:  (663276, 119)
  Target:  escalation_needed (binary, 1 = ICU outcome)
  Leakage: None — cumulative aggregation only uses past timesteps

Split design:
  Split by subject_id - no patient appears in both train and test
  Stratified by cohort_label - class distribution preserved
  Scaler fit on training data only
  Pushing part1_train ((67085, 73))...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]


All datasets pushed to HuggingFace.
Repo: ADS599-Capstone/modeling_data
Configs: part1_train, part1_test, part2_train, part2_test
